# 06 — Data audit, sensitivity analysis and validation decision

**Goal:** finish the validation. Re-check the data and target with automated tests, stress the
model's inputs, parameters and modelling choices, then score it on ten dimensions and apply
decision rules fixed in advance (`src/pdmodel/assessment.py`).

Everything here uses the recalibrated model from notebook 05 on 2015 loans.

In [1]:
import os, sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pdmodel import config as C
pd.set_option("display.float_format", "{:.4f}".format)
PROC = ROOT / "data" / "processed"
from pdmodel import assessment as A, audit, data, model, sensitivity as SE, validation as V

In [2]:
S = pd.read_pickle(PROC / "samples.pkl")
M = pd.read_pickle(PROC / "model.pkl")
VAL = pd.read_pickle(PROC / "validation.pkl")
card, final, cm = M["scorecard"], M["features"], VAL["calibrated_model"]
oot, y_oot = S["oot"], S["oot"]["default"].to_numpy()
p_oot = cm.predict_pd(oot)

## 1. Data audit

The audit needs the raw file, to see every loan status, including the loans excluded as unresolved.

In [3]:
DATA_PATH = os.environ.get("PD_DATA_PATH")   # None -> auto-find in data/raw/
raw = data.load_raw(DATA_PATH)
checks, T = audit.run_checks(raw, data.prepare(raw), S, final)
del raw
audit.checks_table(checks)

,result,evidence,dimension,severity
check,,,,
Fully paid loans are non-defaults,PASS,"Both 'Fully Paid' statuses map to 0, and over ...",target definition,critical
Charged-off and defaulted loans are defaults,PASS,"Charged Off, Default, Does not meet the credit...",target definition,critical
"Unresolved loans excluded, and rare in modelled vintages",PASS,0.02% of 2007–2015 36-month loans unresolved (...,target definition,minor
Every modelled vintage observed for its full 36-month term,PASS,Snapshot Mar 2019; shortest window 39 months,target definition,critical
No post-origination field among the candidate features,PASS,18 candidate features checked against 24 outco...,data quality,critical
Final model uses no post-origination field,PASS,9 model features checked,data quality,critical
Samples separated in time (train/test < calibration < out-of-time),PASS,train 2007-06–2013-12 · test 2007-06–2013-12 ·...,data quality,critical
Out-of-range values below 0.1% of loans,PASS,"9 out-of-range values across 621,022 loans",data quality,minor
Final features at most 10% missing in training,PASS,"Highest: revol_util 0.1%, inq_last_6mths 0.0%",data quality,minor


In [4]:
T["resolution_by_vintage"].style.format({"unresolved_share": "{:.2%}", "default_rate_resolved": "{:.2%}"})

outcome,n_loans,default,non-default,unresolved (excluded),unresolved_share,default_rate_resolved,modelled
issue_year,,,,,,,
2007,603,158,445,0,0.00%,26.20%,True
2008,2393,496,1897,0,0.00%,20.73%,True
2009,5281,723,4558,0,0.00%,13.69%,True
2010,9156,1000,8156,0,0.00%,10.92%,True
2011,14101,1499,12602,0,0.00%,10.63%,True
2012,43470,5903,37567,0,0.00%,13.58%,True
2013,100422,12378,88044,0,0.00%,12.33%,True
2014,162570,22315,140255,0,0.00%,13.73%,True
2015,283173,42132,240894,147,0.05%,14.89%,True


Early repayments count as non-defaults: the borrower repaid the full principal.

In [5]:
T["prepayment"].astype({"fully_paid": int})

,fully_paid,repaid_3m_early_share,median_months_to_payoff,median_months_to_last_payment_defaults
issue_year,,,,
2007,445,0.3416,36.0000,15.0000
2008,1897,0.3843,36.0000,14.0000
2009,4558,0.4774,34.0000,15.0000
2010,8156,0.4881,33.0000,16.0000
2011,12602,0.4944,33.0000,16.0000
2012,37567,0.5164,32.0000,16.0000
2013,88044,0.5717,29.0000,17.0000
2014,140255,0.5871,28.0000,17.0000
2015,240894,0.5822,29.0000,16.0000


## 2. Borrower characteristics

One adverse change applied to every borrower at a time, then all together.

In [6]:
SE.characteristic_shifts(cm, oot)

,mean_pd,relative_change,loans_with_higher_pd,loans_downgraded
change,,,,
Baseline (no change),0.1318,0.0000,0.0000,0.0000
FICO 20 points lower,0.1485,0.1269,0.7668,0.4931
Debt-to-income 5 points higher,0.1370,0.0396,0.7755,0.1603
Income 10% lower,0.1391,0.0555,0.7221,0.2151
Revolving utilisation 10 points higher,0.1343,0.0187,0.8543,0.0725
One more credit inquiry,0.1567,0.1888,0.9587,0.6728
Credit history 2 years shorter,0.1327,0.0067,0.4937,0.0251
All of the above together,0.1967,0.4922,1.0000,0.8286


## 3. Missing inputs

Blank one input for every loan. A numeric input then gets its "Missing" bin's WoE; look at how
many training loans each of those bins was fitted on.

In [7]:
SE.missing_inputs(cm, oot, y_oot)

,woe_used,gini,mean_pd,gini_change,mean_pd_change
input_removed,,,,,
None (baseline),NaN,0.2961,0.1318,0.0000,0.0000
fico,0.0000,0.2504,0.1241,-0.0456,-0.0585
annual_inc,0.2632,0.2938,0.1143,-0.0023,-0.1329
loan_to_income,0.2632,0.2890,0.1117,-0.0071,-0.1522
inq_last_6mths,0.2632,0.2698,0.1073,-0.0263,-0.1860
purpose,-0.2562,0.2878,0.1666,-0.0082,0.2642
dti,0.0000,0.2863,0.1288,-0.0097,-0.0228
revol_util,-0.2442,0.2967,0.1414,0.0006,0.0731
home_ownership,-0.3898,0.2905,0.1510,-0.0056,0.1457


In [8]:
w = card.binner.woe_table(final)
w[w["bin"] == "Missing"][["feature", "n", "bads", "woe"]]

,feature,n,bads,woe
20,annual_inc,4.0000,0.0000,0.2632
29,loan_to_income,4.0000,0.0000,0.2632
34,inq_last_6mths,24.0000,2.0000,0.2632
64,revol_util,137.0000,21.0000,-0.2442
76,credit_hist_years,24.0000,2.0000,0.2632


## 4. Coefficients

Each coefficient moved by ±1 standard error (largest effects first), then whole coefficient
vectors drawn from their estimated covariance.

In [9]:
single, joint = SE.coefficient_sensitivity(cm, oot, y_oot, n_draws=500)
single.reindex(single["mean_pd_change"].abs().sort_values(ascending=False).index).head(6)

mean_pd  mean_pd_change   gini
coefficient shift                                
const       +1 SE   0.1328          0.0075 0.2961
            -1 SE   0.1308         -0.0075 0.2961
fico        -1 SE   0.1322          0.0030 0.2962
            +1 SE   0.1314         -0.0029 0.2959
dti         -1 SE   0.1322          0.0026 0.2966
            +1 SE   0.1315         -0.0026 0.2955

In [10]:
joint

,base,p2.5,p97.5,relative_half_width
mean_pd,0.1318,0.1298,0.1340,0.0160
gini,0.2961,0.2932,0.2982,0.0086


## 5. Sub-populations

Performance where the borrower mix differs from the average.

In [11]:
SE.segment_performance(y_oot, p_oot, SE.default_segments(oot))

,n,share,mean_pd,observed_dr,eo_ratio,gini
segment,,,,,,
All loans,283026,1.0000,0.1318,0.1489,0.8854,0.2961
FICO below 680,106532,0.3764,0.1697,0.1932,0.8785,0.2037
FICO 740 and above,26067,0.0921,0.0551,0.0602,0.9156,0.3495
Debt-to-income above 25,66763,0.2359,0.1623,0.1947,0.8339,0.2347
Income below $40k,52432,0.1853,0.1919,0.1985,0.9668,0.2099
Income $100k and above,55786,0.1971,0.0843,0.1080,0.7807,0.3288
Renters,121463,0.4292,0.1528,0.1765,0.8656,0.2509
Mortgage holders,129762,0.4585,0.1100,0.1206,0.9117,0.3071
Debt consolidation,160983,0.5688,0.1407,0.1573,0.8942,0.2855


## 6. Probability thresholds

Approve when PD is below the cut-off.

In [12]:
SE.threshold_table(y_oot, p_oot)

,approval_rate,bad_rate_approved,bad_rate_declined,defaults_avoided,goods_declined
pd_cutoff,,,,,
0.0500,0.0665,0.0365,0.1569,0.9837,0.9247
0.0800,0.2210,0.0637,0.1730,0.9054,0.7569
0.1000,0.3551,0.0803,0.1866,0.8085,0.6162
0.1200,0.4884,0.0942,0.2011,0.6909,0.4802
0.1500,0.6621,0.1109,0.2232,0.5067,0.3084
0.2000,0.8553,0.1303,0.2584,0.2513,0.1261
0.2500,0.9481,0.1410,0.2932,0.1023,0.0431
0.3000,0.9846,0.1462,0.3222,0.0333,0.0123


## 7. Alternative modelling assumptions

The whole development (binning, selection, fit) re-run with one choice changed, recalibrated on
2014 the same way, and scored on 2015. Takes about a minute.

In [13]:
alts = SE.alternative_specifications(S, VAL["recalibration"]["method"])
alts.drop(columns="features")

,train_loans,n_features,gini_oot,brier_oot,eo_ratio_oot,cal_slope_oot
specification,,,,,,
Baseline (as developed),122798,9,0.2961,0.1227,0.8854,0.9546
At most 5 starting bins,122798,8,0.2920,0.1228,0.8841,0.9541
Minimum bin size 10%,122798,9,0.2925,0.1228,0.8857,0.9567
No monotonic constraint,122798,9,0.2960,0.1227,0.8855,0.9541
Keep mostly-missing features (mort_acc),122798,9,0.2966,0.1227,0.8874,0.9443
Train on all 2007–13 loans,175426,8,0.2946,0.1227,0.8854,0.9627
Train on 2007–12 only (chronological),75004,7,0.2818,0.1231,0.8777,0.9357


## 8. Validation assessment and decision

The rules, fixed before the assessment was run:

In [14]:
A.rules_table()

,criterion,rationale
decision,,
Reject,"RED in conceptual soundness, data quality or t...",A flaw in what the model is or what it learns ...
Redevelop,"RED in population stability, out-of-time perfo...","The ranking itself is broken, unstable or clea..."
Recalibrate,"RED in calibration, with none of the above",The ranking is sound and only the PD level is ...
Conditionally approve,"No RED, except discrimination when no challeng...",Fit for use with stated conditions and monitoring
Approve,GREEN in dimensions 1–9 (limitations may be AM...,No conditions beyond standard monitoring


In [15]:
coefs = model.coefficient_table(M["logit"], card.binner.transform(S["train"], final))
y = {k: v["default"].to_numpy() for k, v in S.items()}
sc = {k: card.score(v) for k, v in S.items()}
metrics = pd.DataFrame({k: {"gini": V.discrimination(y[k], card.predict_pd(S[k]))["gini"]} for k in ("train", "oot")}).T
metrics.loc["oot", "score_psi_vs_train"] = V.psi(sc["train"], sc["oot"])
evidence = A.gather_evidence(
    checks, coefs, final, card.binner, metrics, V.csi_by_feature(card.binner, S["train"], oot, final),
    V.rank_ordering(y_oot, sc["oot"]), VAL["ci"], VAL["paired"], {"alternatives": alts, "coef_joint": joint},
    A.read_register(ROOT / "reports" / "limitations_register.md"))
dims, decisions = A.assess_versions(evidence, y_oot, {"as developed": card.predict_pd(oot), "recalibrated": p_oot},
                                    cm.grade(oot))
A.dimensions_table(dims["recalibrated"])

,dimension,status,evidence
#,,,
1,Conceptual soundness,GREEN,0 coefficients with the wrong sign; largest p-...
2,Data quality,GREEN,5 of 5 checks passed.
3,Target definition,GREEN,4 of 4 checks passed.
4,Model discrimination,RED,Out-of-time Gini 0.296 (95% CI 0.292–0.300); t...
5,Model calibration,RED,Mean PD minus observed default rate -1.71%; 6 ...
6,Population stability,GREEN,Score PSI 0.001; largest feature CSI 0.098
7,Out-of-time performance,GREEN,Gini falls 6.2% from train to out-of-time; ran...
8,Challenger comparison,GREEN,Challenger Gini minus scorecard +0.022 (95% CI...
9,Sensitivity analysis,GREEN,Out-of-time Gini range across alternative spec...


In [16]:
pd.DataFrame(decisions, index=["decision", "reason"]).T

,decision,reason
as developed,Recalibrate,RED calibration with a sound ranking
recalibrated,Recalibrate,RED calibration with a sound ranking


## Findings
- All nine audit checks pass. Only 147 loans (0.02%) in the modelled vintages were unresolved at the
  March 2019 snapshot, and 57% of fully paid loans were repaid at least three months early.
- Every adverse change raises the PD in the expected direction: one more credit inquiry +18.9%,
  FICO 20 points lower +12.7%, all six changes together +49%.
- **Weakness:** a blank `annual_inc`, `loan_to_income` or `inq_last_6mths` *lowers* the PD by 13–19%.
  Their "Missing" bins were fitted on 4–24 training loans, almost none of which defaulted.
- Coefficient uncertainty moves the mean PD by only ±1.6%, and seven alternative developments keep
  the out-of-time Gini between 0.282 and 0.297: the model is not fragile to development choices.
- Ranking is weakest for the riskiest borrowers (Gini 0.20 for FICO below 680), and PDs are most
  understated for loans of $25k and above (expected-to-observed ratio 0.76).
- Decision: **Recalibrate** for both versions. The ranking dimensions are GREEN, but the 2014
  recalibration still fails calibration on 2015, so it must be refreshed on more recent data and
  shown to hold out of time. Discrimination is RED against the 0.30 threshold, but no challenger
  on the same data is materially better, so it does not trigger redevelopment.